In [ ]:
from google.colab import drive
import os

# Ensure the mount point is empty
if os.path.exists('/content/drive'):
    !rm -rf /content/drive/*

drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
!pip install datasets huggingface_hub -q

from huggingface_hub import login
from google.colab import userdata
from huggingface_hub import login

token = userdata.get('HF_TOKEN')
login(token=token)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("uvci/Koumankan_mt_dyu_fr")
print(dataset)

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/530k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/102k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/55.8k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8065 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1471 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1393 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['ID', 'translation'],
        num_rows: 8065
    })
    validation: Dataset({
        features: ['ID', 'translation'],
        num_rows: 1471
    })
    test: Dataset({
        features: ['ID', 'translation'],
        num_rows: 1393
    })
})


In [ ]:
import os
os.makedirs("/content/drive/MyDrive/kalan/dataset", exist_ok=True)

# Sauvegarder
dataset.save_to_disk("/content/drive/MyDrive/kalan/dataset/koumankan")
print(" Dataset sauvegardé dans Google Drive !")

Saving the dataset (0/1 shards):   0%|          | 0/8065 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1471 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1393 [00:00<?, ? examples/s]

 Dataset sauvegardé dans Google Drive !


In [ ]:
for i in range(3):
    exemple = dataset["train"][i]
    print(f"ID: {exemple['ID']}")
    print(f"Traduction: {exemple['translation']}")
    print("---")


ID: ID_18897661270129
Traduction: {'dyu': 'A bi ji min na', 'fr': 'Il boit de l’eau.'}
---
ID: ID_18479132727846
Traduction: {'dyu': 'A le dalakolontɛ lon bɛ.', 'fr': 'Il se plaint toujours.'}
---
ID: ID_18164131280307
Traduction: {'dyu': 'Mun? Fɛn dɔ.', 'fr': 'Quoi ? Quelque chose.'}
---


In [ ]:
# Installer les librairies nécessaires
!pip install transformers sentencepiece sacrebleu -q

from transformers import MarianMTModel, MarianTokenizer

# Charger le modèle de traduction
model_name = "Helsinki-NLP/opus-mt-tc-big-fr-en"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

print("Modèle chargé !")
print(f"Paramètres : {model.num_parameters():,}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.6 MB/s eta 0:00:00


tokenizer_config.json:   0%|          | 0.00/337 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/820k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/461M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

Modèle chargé !
Paramètres : 232,743,936


In [ ]:
# Tester le modèle AVANT fine-tuning
phrases_test = [
    "Bonjour, comment vas-tu ?",
    "Je veux apprendre le Dioula.",
    "L'eau est froide."
]

def traduire(texte):
    inputs = tokenizer(texte, return_tensors="pt", padding=True)
    outputs = model.generate(**inputs)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print("=== Traductions AVANT fine-tuning ===\n")
for phrase in phrases_test:
    print(f"Français  : {phrase}")
    print(f"Traduction: {traduire(phrase)}")
    print("---")


=== Traductions AVANT fine-tuning ===

Français  : Bonjour, comment vas-tu ?
Traduction: Hello, how are you?
---
Français  : Je veux apprendre le Dioula.
Traduction: I want to learn Dioula.
---
Français  : L'eau est froide.
Traduction: The water is cold.
---


In [ ]:
from datasets import load_from_disk

# Charger le dataset sauvegardé dans Drive
dataset = load_from_disk("/content/drive/MyDrive/kalan/dataset/koumankan")

# Reformater pour le modèle MarianMT
def preprocess(exemples):
    inputs = [ex["fr"] for ex in exemples["translation"]]
    targets = [ex["dyu"] for ex in exemples["translation"]]

    model_inputs = tokenizer(
        inputs,
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        targets,
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Appliquer le formatage
dataset_tokenise = dataset.map(preprocess, batched=True)
print("Données prêtes !")
print(dataset_tokenise)

Données prêtes !
DatasetDict({
    train: Dataset({
        features: ['ID', 'translation', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 8065
    })
    validation: Dataset({
        features: ['ID', 'translation', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1471
    })
    test: Dataset({
        features: ['ID', 'translation', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1393
    })
})


In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
import os

# Dossier de sauvegarde du modèle
output_dir = "/content/drive/MyDrive/kalan/modele_traduction"
os.makedirs(output_dir, exist_ok=True)

# Set the environment variable for TensorBoard logging
os.environ["TENSORBOARD_LOGGING_DIR"] = f"{output_dir}/logs"

# Configuration de l'entraînement
training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    predict_with_generate=True,
    fp16=False,  # Désactiver fp16 pour éviter l'erreur de dé-scaling des gradients
)

# Explicitly cast model to float32 to ensure no FP16 gradients are generated
model.float()

# Créer un DataCollator pour le modèle
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# Créer le trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset_tokenise["train"],
    eval_dataset=dataset_tokenise["validation"],
    data_collator=data_collator,
)

print("Trainer prêt — lancement de l'entraînement...")
trainer.train()
print("Entraînement terminé !")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Trainer prêt — lancement de l'entraînement...


Epoch,Training Loss,Validation Loss
1,0.394895,0.385973
2,0.311161,0.318097
3,0.255357,0.309885


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.encoder.embed_positions.weight', 'model.decoder.embed_tokens.weight', 'model.decoder.embed_positions.weight', 'lm_head.weight'].


Entraînement terminé !


In [ ]:
# Redéfinir les phrases de test
phrases_test = [
    "Bonjour, comment vas-tu ?",
    "Je veux apprendre le Dioula.",
    "L'eau est froide."
]

# Tester le modèle APRÈS fine-tuning
def traduire(texte):
    inputs = tokenizer(texte, return_tensors="pt", padding=True)
    outputs = model.generate(**inputs)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print("=== Traductions APRÈS fine-tuning ===\n")
for phrase in phrases_test:
    print(f"Français  : {phrase}")
    print(f"Dioula    : {traduire(phrase)}")
    print("---")

=== Traductions APRÈS fine-tuning ===

Français  : Bonjour, comment vas-tu ?
Dioula    : Hello, how are you?
---
Français  : Je veux apprendre le Dioula.
Dioula    : I want to learn Dioula.
---
Français  : L'eau est froide.
Dioula    : The water is cold.
---


In [ ]:
import os

# Charger le modèle fine-tuné depuis Drive
output_dir = "/content/drive/MyDrive/kalan/modele_traduction"

# Vérifier que le modèle est bien sauvegardé
print("Fichiers sauvegardés :")
if os.path.exists(output_dir):
    for f in os.listdir(output_dir):
        print(f" -", f)
else:
    print(f"Le dossier '{output_dir}' n'existe pas. Le modèle n'a peut-être pas été sauvegardé ou le chemin est incorrect.")

Fichiers sauvegardés :
Le dossier '/content/drive/MyDrive/kalan/modele_traduction' n'existe pas. Le modèle n'a peut-être pas été sauvegardé ou le chemin est incorrect.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install transformers sentencepiece sacrebleu datasets huggingface_hub -q

from google.colab import userdata
from huggingface_hub import login
token = userdata.get('HF_TOKEN')
login(token=token)

print("Environnement prêt !")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.2 MB/s eta 0:00:00
Environnement prêt !


In [ ]:
from transformers import MarianMTModel, MarianTokenizer

model_name = "Helsinki-NLP/opus-mt-tc-big-fr-en"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

print("Modèle chargé !")

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Modèle chargé !


In [ ]:
from datasets import load_from_disk

dataset = load_from_disk("/content/drive/MyDrive/kalan/dataset/koumankan")

def preprocess(exemples):
    inputs = [ex["fr"] for ex in exemples["translation"]]
    targets = [ex["dyu"] for ex in exemples["translation"]]

    model_inputs = tokenizer(
        inputs, max_length=128,
        truncation=True, padding="max_length"
    )
    labels = tokenizer(
        targets, max_length=128,
        truncation=True, padding="max_length"
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

dataset_tokenise = dataset.map(preprocess, batched=True)
print("Données prêtes !")

Map:   0%|          | 0/8065 [00:00<?, ? examples/s]

Map:   0%|          | 0/1471 [00:00<?, ? examples/s]

Map:   0%|          | 0/1393 [00:00<?, ? examples/s]

Données prêtes !


In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
import os

# Créer le dossier de sauvegarde
output_dir = "/content/drive/MyDrive/kalan/modele_traduction"
os.makedirs(output_dir, exist_ok=True)
print(f"✅ Dossier créé : {output_dir}")

# Vérifier que le dossier existe bien
print(f"Dossier existe : {os.path.exists(output_dir)}")

# Configuration de l'entraînement
training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    predict_with_generate=True,
    fp16=True,
)

# Créer le trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset_tokenise["train"],
    eval_dataset=dataset_tokenise["validation"],
    tokenizer=tokenizer,
)

print(" Lancement de l'entraînement...")
trainer.train()

# Sauvegarder explicitement après l'entraînement
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(" Entraînement terminé et modèle sauvegardé !")

✅ Dossier créé : /content/drive/MyDrive/kalan/modele_traduction
Dossier existe : True


TypeError: Seq2SeqTrainer.__init__() got an unexpected keyword argument 'tokenizer'

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
import os

# Créer le dossier de sauvegarde
output_dir = "/content/drive/MyDrive/kalan/modele_traduction"
os.makedirs(output_dir, exist_ok=True)
print(f" Dossier créé : {output_dir}")

# Vérifier que le dossier existe bien
print(f"Dossier existe : {os.path.exists(output_dir)}")

# Configuration de l'entraînement
training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    predict_with_generate=True,
    fp16=False,  # Changed to False to prevent FP16 gradient unscaling error
)

# Create a DataCollator for the model
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# Créer le trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset_tokenise["train"],
    eval_dataset=dataset_tokenise["validation"],
    data_collator=data_collator,
)

print(" Lancement de l'entraînement...")
trainer.train()

# Sauvegarder explicitement après l'entraînement
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(" Entraînement terminé et modèle sauvegardé !")

 Dossier créé : /content/drive/MyDrive/kalan/modele_traduction
Dossier existe : True
 Lancement de l'entraînement...


Epoch,Training Loss,Validation Loss
1,0.000000,nan


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# Vérifier les données tokenisées
exemple = dataset_tokenise["train"][0]
print("Clés disponibles :", exemple.keys())
print("input_ids :", exemple["input_ids"][:10])
print("labels :", exemple["labels"][:10])

# Vérifier s'il y a des valeurs nulles
import numpy as np
labels = dataset_tokenise["train"]["labels"]
nan_count = sum(1 for l in labels if l is None)
print(f"Labels nuls : {nan_count}")

Clés disponibles : dict_keys(['ID', 'translation', 'input_ids', 'attention_mask', 'labels'])
input_ids : [25402, 7811, 14012, 28665, 53, 16872, 35, 43311, 53016, 53016]
labels : [1896, 7384, 104, 28006, 31884, 32949, 1893, 43311, 53016, 53016]
Labels nuls : 0


In [ ]:
from transformers import DataCollatorForSeq2Seq

# Corriger le data collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,  # ← important pour ignorer le padding
    pad_to_multiple_of=8
)

# Recréer le trainer corrigé
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset_tokenise["train"],
    eval_dataset=dataset_tokenise["validation"],
    data_collator=data_collator,
)

print(" Relancement de l'entraînement...")
trainer.train()

model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(" Entraînement terminé et modèle sauvegardé !")

 Relancement de l'entraînement...


Epoch,Training Loss,Validation Loss
1,0.000000,nan
2,0.000000,nan
3,0.000000,nan


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

 Entraînement terminé et modèle sauvegardé !


In [ ]:
from transformers import MarianMTModel, MarianTokenizer

# Charger le modèle fine-tuné
output_dir = "/content/drive/MyDrive/kalan/modele_traduction"
tokenizer_ft = MarianTokenizer.from_pretrained(output_dir)
model_ft = MarianMTModel.from_pretrained(output_dir)

# Tester
phrases_test = [
    "Bonjour, comment vas-tu ?",
    "Je veux apprendre le Dioula.",
    "L'eau est froide."
]

def traduire_dioula(texte):
    inputs = tokenizer_ft(texte, return_tensors="pt", padding=True)
    outputs = model_ft.generate(**inputs, max_length=128)
    return tokenizer_ft.decode(outputs[0], skip_special_tokens=True)

print("=== Traductions APRÈS fine-tuning ===\n")
for phrase in phrases_test:
    print(f"Français : {phrase}")
    print(f"Dioula   : {traduire_dioula(phrase)}")
    print("---")

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

=== Traductions APRÈS fine-tuning ===

Français : Bonjour, comment vas-tu ?
Dioula   : >
---
Français : Je veux apprendre le Dioula.
Dioula   : >
---
Français : L'eau est froide.
Dioula   : >
---


In [ ]:
def traduire_dioula(texte):
    inputs = tokenizer_ft(texte, return_tensors="pt", padding=True)
    outputs = model_ft.generate(
        **inputs,
        max_length=128,
        num_beams=4,           # recherche par faisceau
        early_stopping=True,
        forced_bos_token_id=tokenizer_ft.convert_tokens_to_ids(">>dyu<<")
        if ">>dyu<<" in tokenizer_ft.get_vocab() else None
    )
    return tokenizer_ft.decode(outputs[0], skip_special_tokens=True)

print("=== Test corrigé ===\n")
for phrase in phrases_test:
    print(f"Français : {phrase}")
    print(f"Dioula   : {traduire_dioula(phrase)}")
    print("---")

=== Test corrigé ===

Français : Bonjour, comment vas-tu ?
Dioula   : >
---
Français : Je veux apprendre le Dioula.
Dioula   : >
---
Français : L'eau est froide.
Dioula   : >
---


In [ ]:
# Vérifier le vocabulaire du tokenizer
print("Taille vocabulaire :", tokenizer_ft.vocab_size)
print("Quelques tokens Dioula dans le vocab ?")

mots_dioula = ["dyu", "kɛ", "bɛ", "ɔ", "ɛ"]
for mot in mots_dioula:
    ids = tokenizer_ft.encode(mot)
    print(f"  '{mot}' → {ids}")

Taille vocabulaire : 53017
Quelques tokens Dioula dans le vocab ?
  'dyu' → [13838, 52901, 43311]
  'kɛ' → [28314, 50387, 43311]
  'bɛ' → [6618, 50387, 43311]
  'ɔ' → [104, 50387, 43311]
  'ɛ' → [104, 50387, 43311]


In [ ]:
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast

print("Chargement du nouveau modèle...")

model_name = "facebook/mbart-large-50-many-to-many-mmt"
tokenizer2 = MBart50TokenizerFast.from_pretrained(model_name)
model2 = MBartForConditionalGeneration.from_pretrained(model_name)

# Définir la langue source
tokenizer2.src_lang = "fr_XX"

print("✅ Nouveau modèle chargé !")
print(f"Langues supportées : {len(tokenizer2.lang_code_to_id)} langues")

# Vérifier les caractères Dioula
mots_dioula = ["kɛ", "bɛ", "ɔ", "ɛ"]
print("\nTokens Dioula avec mbart :")
for mot in mots_dioula:
    ids = tokenizer2.encode(mot)
    print(f"  '{mot}' → {ids}")

Chargement du nouveau modèle...


tokenizer_config.json:   0%|          | 0.00/529 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

✅ Nouveau modèle chargé !
Langues supportées : 52 langues

Tokens Dioula avec mbart :
  'kɛ' → [250008, 472, 171522, 2]
  'bɛ' → [250008, 876, 171522, 2]
  'ɔ' → [250008, 6, 184135, 2]
  'ɛ' → [250008, 6, 171522, 2]


In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

print("Chargement de NLLB...")

model_name = "facebook/nllb-200-distilled-600M"
tokenizer3 = AutoTokenizer.from_pretrained(model_name)
model3 = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Vérifier que le Dioula est supporté
dioula_token = "dyu_Latn"
print(f"\nDioula dans le vocab : {dioula_token in tokenizer3.lang_code_to_id}")

# Tester une traduction directement sans fine-tuning
tokenizer3.src_lang = "fra_Latn"
inputs = tokenizer3("Bonjour, comment vas-tu ?", return_tensors="pt")
outputs = model3.generate(
    **inputs,
    forced_bos_token_id=tokenizer3.lang_code_to_id[dioula_token],
    max_length=128
)
print("\nTest traduction :")
print("Français :", "Bonjour, comment vas-tu ?")
print("Dioula   :", tokenizer3.decode(outputs[0], skip_special_tokens=True))

Chargement de NLLB...


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

AttributeError: TokenizersBackend has no attribute lang_code_to_id

In [3]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

print("Chargement de NLLB...")

model_name = "facebook/nllb-200-distilled-600M"
tokenizer3 = AutoTokenizer.from_pretrained(model_name)
model3 = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Vérifier que le Dioula est supporté
dioula_token = "dyu_Latn"
# NLLB language tokens are typically formatted as __lang_code__
dioula_token_formatted = f"__{dioula_token}__"

# Check if the formatted language token is in the tokenizer's vocabulary
try:
    dioula_token_id = tokenizer3.convert_tokens_to_ids(dioula_token_formatted)
    # A token ID different from the unknown token ID means it's supported
    is_dioula_supported = (dioula_token_id != tokenizer3.unk_token_id)
except KeyError: # convert_tokens_to_ids can raise KeyError if token not found
    is_dioula_supported = False
print(f"\nDioula dans le vocab : {is_dioula_supported}")

# Tester une traduction directement sans fine-tuning
tokenizer3.src_lang = "fra_Latn"
inputs = tokenizer3("Bonjour, comment vas-tu ?", return_tensors="pt")
outputs = model3.generate(
    **inputs,
    forced_bos_token_id=tokenizer3.convert_tokens_to_ids(dioula_token_formatted),
    max_length=128
)
print("\nTest traduction :")
print("Français :", "Bonjour, comment vas-tu ?")
print("Dioula   :", tokenizer3.decode(outputs[0], skip_special_tokens=True))

Chargement de NLLB...


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]


Dioula dans le vocab : False

Test traduction :
Français : Bonjour, comment vas-tu ?
Dioula   : Bonjour, comment allez-vous ?


In [4]:
# Tester NLLB sans fine-tuning
tokenizer3.src_lang = "fra_Latn"
inputs = tokenizer3("Bonjour, comment vas-tu ?", return_tensors="pt")

# Récupérer l'ID du token Dioula
dioula_id = tokenizer3.convert_tokens_to_ids("dyu_Latn")
print(f"Token ID Dioula : {dioula_id}")

outputs = model3.generate(
    **inputs,
    forced_bos_token_id=dioula_id,
    max_length=128,
    num_beams=4
)

print("Français :", "Bonjour, comment vas-tu ?")
print("Dioula   :", tokenizer3.decode(outputs[0], skip_special_tokens=True))

Token ID Dioula : 256044
Français : Bonjour, comment vas-tu ?
Dioula   : Bonjour, i be cogo di?


In [5]:
phrases_test = [
    "Bonjour, comment vas-tu ?",
    "Je veux apprendre le Dioula.",
    "L'eau est froide.",
    "Merci beaucoup.",
    "Je m'appelle Dorcas."
]

print("=Traductions NLLB (sans fine-tuning)=\n")
for phrase in phrases_test:
    inputs = tokenizer3(phrase, return_tensors="pt")
    outputs = model3.generate(
        **inputs,
        forced_bos_token_id=dioula_id,
        max_length=128,
        num_beams=4
    )
    traduction = tokenizer3.decode(outputs[0], skip_special_tokens=True)
    print(f"Français : {phrase}")
    print(f"Dioula   : {traduction}")
    print("---")

=Traductions NLLB (sans fine-tuning)=

Français : Bonjour, comment vas-tu ?
Dioula   : Bonjour, i be cogo di?
---
Français : Je veux apprendre le Dioula.
Dioula   : Ne b'a fɛ ka Dioula degi.
---
Français : L'eau est froide.
Dioula   : Jii be sumaya.
---
Français : Merci beaucoup.
Dioula   : Ne b'i fo kosɔbɛ.
---
Français : Je m'appelle Dorcas.
Dioula   : Ne tɔgɔ ko Dorkasi.
---


In [9]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
from transformers import DataCollatorForSeq2Seq
from datasets import load_from_disk # Import load_from_disk
import os

# Préparer les données pour NLLB
tokenizer3.src_lang = "fra_Latn"
target_lang_id = tokenizer3.convert_tokens_to_ids("dyu_Latn")

# Load the dataset (assuming it's saved in Drive as per previous steps)
dataset = load_from_disk("/content/drive/MyDrive/kalan/dataset/koumankan")

def preprocess_nllb(exemples):
    inputs = [ex["fr"] for ex in exemples["translation"]]
    targets = [ex["dyu"] for ex in exemples["translation"]]

    model_inputs = tokenizer3(
        inputs,
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    with tokenizer3.as_target_tokenizer():
        labels = tokenizer3(
            targets,
            max_length=128,
            truncation=True,
            padding="max_length"
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

dataset_nllb = dataset.map(preprocess_nllb, batched=True)
print(" Données préparées pour NLLB !")
print(dataset_nllb)

FileNotFoundError: Directory /content/drive/MyDrive/kalan/dataset/koumankan not found

In [10]:
from datasets import load_dataset

# Recharger depuis Hugging Face
dataset = load_dataset("uvci/Koumankan_mt_dyu_fr")
print(" Dataset rechargé !")
print(dataset)

# Sauvegarder dans Drive pour ne plus avoir ce problème
dataset.save_to_disk("/content/drive/MyDrive/kalan/dataset/koumankan")
print("Dataset sauvegardé dans Drive !")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/530k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/102k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/55.8k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8065 [00:00<?, ? examples/s]

...

 Dataset rechargé !
DatasetDict({
    train: Dataset({
        features: ['ID', 'translation'],
        num_rows: 8065
    })
    validation: Dataset({
        features: ['ID', 'translation'],
        num_rows: 1471
    })
    test: Dataset({
        features: ['ID', 'translation'],
        num_rows: 1393
    })
})


Saving the dataset (0/1 shards):   0%|          | 0/8065 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1471 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1393 [00:00<?, ? examples/s]

Dataset sauvegardé dans Drive !


In [11]:
tokenizer3.src_lang = "fra_Latn"
target_lang_id = tokenizer3.convert_tokens_to_ids("dyu_Latn")

def preprocess_nllb(exemples):
    inputs = [ex["fr"] for ex in exemples["translation"]]
    targets = [ex["dyu"] for ex in exemples["translation"]]

    model_inputs = tokenizer3(
        inputs,
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    with tokenizer3.as_target_tokenizer():
        labels = tokenizer3(
            targets,
            max_length=128,
            truncation=True,
            padding="max_length"
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

dataset_nllb = dataset.map(preprocess_nllb, batched=True)
print(" Données préparées pour NLLB !")
print(dataset_nllb)

Map:   0%|          | 0/8065 [00:00<?, ? examples/s]

AttributeError: TokenizersBackend has no attribute as_target_tokenizer

In [13]:
tokenizer3.src_lang = "fra_Latn"
target_lang_id = tokenizer3.convert_tokens_to_ids("dyu_Latn")

def preprocess_nllb(exemples):
    inputs = [ex["fr"] for ex in exemples["translation"]]
    targets = [ex["dyu"] for ex in exemples["translation"]]

    model_inputs = tokenizer3(
        inputs,
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    # Version corrigée — sans as_target_tokenizer
    labels = tokenizer3(
        text_target=targets,
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

dataset_nllb = dataset.map(preprocess_nllb, batched=True)
print(" Données préparées pour NLLB !")
print(dataset_nllb)

 Données préparées pour NLLB !
DatasetDict({
    train: Dataset({
        features: ['ID', 'translation', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 8065
    })
    validation: Dataset({
        features: ['ID', 'translation', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1471
    })
    test: Dataset({
        features: ['ID', 'translation', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1393
    })
})


In [14]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
import os

output_dir = "/content/drive/MyDrive/kalan/modele_nllb"
os.makedirs(output_dir, exist_ok=True)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer3,
    model=model3,
    label_pad_token_id=-100,
    pad_to_multiple_of=8
)

training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=8,  # réduit car NLLB est plus grand
    per_device_eval_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    predict_with_generate=True,
    fp16=True,
)

trainer = Seq2SeqTrainer(
    model=model3,
    args=training_args,
    train_dataset=dataset_nllb["train"],
    eval_dataset=dataset_nllb["validation"],
    data_collator=data_collator,
)

print("🚀 Lancement de l'entraînement NLLB...")
trainer.train()

model3.save_pretrained(output_dir)
tokenizer3.save_pretrained(output_dir)
print("🎉 Entraînement terminé et modèle sauvegardé !")

🚀 Lancement de l'entraînement NLLB...


OutOfMemoryError: CUDA out of memory. Tried to allocate 1002.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 665.81 MiB is free. Including non-PyTorch memory, this process has 13.91 GiB memory in use. Of the allocated memory 13.55 GiB is allocated by PyTorch, and 229.59 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [15]:
import torch
import os

# Vider la mémoire GPU
torch.cuda.empty_cache()

output_dir = "/content/drive/MyDrive/kalan/modele_nllb"
os.makedirs(output_dir, exist_ok=True)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer3,
    model=model3,
    label_pad_token_id=-100,
    pad_to_multiple_of=8
)

training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=2,   # ← réduit à 2
    per_device_eval_batch_size=2,    # ← réduit à 2
    gradient_accumulation_steps=8,  # ← compense le petit batch
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    predict_with_generate=True,
    fp16=True,
    optim="adafactor",              # ← optimiseur léger
)

trainer = Seq2SeqTrainer(
    model=model3,
    args=training_args,
    train_dataset=dataset_nllb["train"],
    eval_dataset=dataset_nllb["validation"],
    data_collator=data_collator,
)

print(" Lancement de l'entraînement NLLB...")
trainer.train()

model3.save_pretrained(output_dir)
tokenizer3.save_pretrained(output_dir)
print(" Entraînement terminé et modèle sauvegardé !")

 Lancement de l'entraînement NLLB...


OutOfMemoryError: CUDA out of memory. Tried to allocate 1002.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 563.81 MiB is free. Including non-PyTorch memory, this process has 14.01 GiB memory in use. Of the allocated memory 13.23 GiB is allocated by PyTorch, and 662.77 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [16]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Vider la mémoire
torch.cuda.empty_cache()
del model3

# Charger la version plus légère
print("Chargement NLLB 350M...")
model_name = "facebook/nllb-200-distilled-350M"
tokenizer3 = AutoTokenizer.from_pretrained(model_name)
model3 = AutoModelForSeq2SeqLM.from_pretrained(model_name)

dioula_id = tokenizer3.convert_tokens_to_ids("dyu_Latn")
print(f"✅ Modèle chargé !")
print(f"Token Dioula : {dioula_id}")

# Tester rapidement
tokenizer3.src_lang = "fra_Latn"
inputs = tokenizer3("Bonjour, comment vas-tu ?", return_tensors="pt")
outputs = model3.generate(
    **inputs,
    forced_bos_token_id=dioula_id,
    max_length=128,
    num_beams=4
)
print("Test :", tokenizer3.decode(outputs[0], skip_special_tokens=True))

Chargement NLLB 350M...


OSError: facebook/nllb-200-distilled-350M is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

In [20]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Vider la mémoire
torch.cuda.empty_cache()
# del model3 # Removed this line as model3 might not be defined

# Nom corrigé
print("Chargement NLLB 350M...")
model_name = "facebook/nllb-200-3.3B"
tokenizer3 = AutoTokenizer.from_pretrained(model_name)
model3 = AutoModelForSeq2SeqLM.from_pretrained(model_name)

dioula_id = tokenizer3.convert_tokens_to_ids("dyu_Latn")
print(f"✅ Modèle chargé !")

# Tester rapidement
tokenizer3.src_lang = "fra_Latn"
inputs = tokenizer3("Bonjour, comment vas-tu ?", return_tensors="pt")
outputs = model3.generate(
    **inputs,
    forced_bos_token_id=dioula_id,
    max_length=128,
    num_beams=4
)
print("Test :", tokenizer3.decode(outputs[0], skip_special_tokens=True))

Chargement NLLB 350M...


Loading weights:   0%|          | 0/1016 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


✅ Modèle chargé !


KeyboardInterrupt: 

In [21]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Vider la mémoire
torch.cuda.empty_cache()
del model3

# Revenir au NLLB 600M
print("Chargement NLLB 600M...")
model_name = "facebook/nllb-200-distilled-600M"
tokenizer3 = AutoTokenizer.from_pretrained(model_name)
model3 = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16  # ← réduit la mémoire de moitié
)

# Installer PEFT pour LoRA
!pip install peft -q

from peft import get_peft_model, LoraConfig, TaskType

# Configurer LoRA
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"]
)

# Appliquer LoRA au modèle
model3 = get_peft_model(model3, lora_config)
model3.print_trainable_parameters()

Chargement NLLB 600M...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


trainable params: 2,359,296 || all params: 1,404,497,920 || trainable%: 0.1680


In [22]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
import os

# Préparer les données
tokenizer3.src_lang = "fra_Latn"
dioula_id = tokenizer3.convert_tokens_to_ids("dyu_Latn")

def preprocess_nllb(exemples):
    inputs = [ex["fr"] for ex in exemples["translation"]]
    targets = [ex["dyu"] for ex in exemples["translation"]]

    model_inputs = tokenizer3(
        inputs, max_length=128,
        truncation=True, padding="max_length"
    )
    labels = tokenizer3(
        text_target=targets,
        max_length=128,
        truncation=True, padding="max_length"
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

dataset_nllb = dataset.map(preprocess_nllb, batched=True)

# Entraînement
output_dir = "/content/drive/MyDrive/kalan/modele_nllb_lora"
os.makedirs(output_dir, exist_ok=True)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer3,
    model=model3,
    label_pad_token_id=-100,
    pad_to_multiple_of=8
)

training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=200,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    predict_with_generate=True,
    fp16=True,
    optim="adafactor",
)

trainer = Seq2SeqTrainer(
    model=model3,
    args=training_args,
    train_dataset=dataset_nllb["train"],
    eval_dataset=dataset_nllb["validation"],
    data_collator=data_collator,
)

print("🚀 Lancement entraînement avec LoRA...")
trainer.train()

model3.save_pretrained(output_dir)
tokenizer3.save_pretrained(output_dir)
print("🎉 Entraînement terminé et modèle sauvegardé !")

Map:   0%|          | 0/8065 [00:00<?, ? examples/s]

Map:   0%|          | 0/1471 [00:00<?, ? examples/s]

Map:   0%|          | 0/1393 [00:00<?, ? examples/s]

🚀 Lancement entraînement avec LoRA...


Epoch,Training Loss,Validation Loss
1,34.398835,8.123674
2,30.751675,7.451574
3,30.392949,7.408626


🎉 Entraînement terminé et modèle sauvegardé !


In [23]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
import os

output_dir = "/content/drive/MyDrive/kalan/modele_nllb_lora"
os.makedirs(output_dir, exist_ok=True)

# Repréparer les données pour ce tokenizer
tokenizer3.src_lang = "fra_Latn"
dioula_id = tokenizer3.convert_tokens_to_ids("dyu_Latn")

def preprocess_nllb(exemples):
    inputs = [ex["fr"] for ex in exemples["translation"]]
    targets = [ex["dyu"] for ex in exemples["translation"]]
    model_inputs = tokenizer3(
        inputs, max_length=128, truncation=True, padding="max_length"
    )
    labels = tokenizer3(
        text_target=targets, max_length=128, truncation=True, padding="max_length"
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

dataset_nllb = dataset.map(preprocess_nllb, batched=True)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer3,
    model=model3,
    label_pad_token_id=-100,
    pad_to_multiple_of=8
)

training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=200,
    weight_decay=0.01,
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    fp16=True,
    optim="adafactor",
    predict_with_generate=False,
)

trainer = Seq2SeqTrainer(
    model=model3,
    args=training_args,
    train_dataset=dataset_nllb["train"],
    eval_dataset=dataset_nllb["validation"],
    data_collator=data_collator,
)

print(" Lancement entraînement avec LoRA...")
trainer.train()

model3.save_pretrained(output_dir)
tokenizer3.save_pretrained(output_dir)
print(" Entraînement terminé et modèle sauvegardé !")

Map:   0%|          | 0/8065 [00:00<?, ? examples/s]

Map:   0%|          | 0/1471 [00:00<?, ? examples/s]

Map:   0%|          | 0/1393 [00:00<?, ? examples/s]

 Lancement entraînement avec LoRA...


Epoch,Training Loss,Validation Loss
1,29.952288,7.340267
2,29.753723,7.322174
3,29.696299,7.319217


 Entraînement terminé et modèle sauvegardé !


In [25]:
import torch

# Tester le modèle fine-tuné
model3.eval()
tokenizer3.src_lang = "fra_Latn"
dioula_id = tokenizer3.convert_tokens_to_ids("dyu_Latn")

phrases_test = [
    "Bonjour, comment vas-tu ?",
    "Je veux apprendre le Dioula.",
    "L'eau est froide.",
    "Merci beaucoup.",
    "Je m'appelle Dorcas."
]

def traduire_dioula(texte):
    inputs = tokenizer3(texte, return_tensors="pt")
    with torch.no_grad():
        outputs = model3.generate(
            **inputs,
            forced_bos_token_id=dioula_id,
            max_length=128,
            num_beams=4
        )
    return tokenizer3.decode(outputs[0], skip_special_tokens=True)

print("=== Traductions APRÈS fine-tuning LoRA ===\n")
for phrase in phrases_test:
    print(f"Français : {phrase}")
    print(f"Dioula   : {traduire_dioula(phrase)}")
    print("---")

=== Traductions APRÈS fine-tuning LoRA ===

Français : Bonjour, comment vas-tu ?


RuntimeError: Expected all tensors to be on the same device, but got index is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA__index_select)

In [26]:
import torch

model3.eval()
tokenizer3.src_lang = "fra_Latn"
dioula_id = tokenizer3.convert_tokens_to_ids("dyu_Latn")

phrases_test = [
    "Bonjour, comment vas-tu ?",
    "Je veux apprendre le Dioula.",
    "L'eau est froide.",
    "Merci beaucoup.",
    "Je m'appelle Dorcas."
]

def traduire_dioula(texte):
    inputs = tokenizer3(texte, return_tensors="pt")
    inputs = {k: v.to("cuda") for k, v in inputs.items()}  # ← déplacer sur GPU
    with torch.no_grad():
        outputs = model3.generate(
            **inputs,
            forced_bos_token_id=dioula_id,
            max_length=128,
            num_beams=4
        )
    return tokenizer3.decode(outputs[0], skip_special_tokens=True)

print("=== Traductions APRÈS fine-tuning LoRA ===\n")
for phrase in phrases_test:
    print(f"Français : {phrase}")
    print(f"Dioula   : {traduire_dioula(phrase)}")
    print("---")

=== Traductions APRÈS fine-tuning LoRA ===

Français : Bonjour, comment vas-tu ?
Dioula   : Bonjour, i bɛ cogo di?
---
Français : Je veux apprendre le Dioula.
Dioula   : Ne b'a fɛ ka Dioula kalan.
---
Français : L'eau est froide.
Dioula   : Ji ye nɛnɛ ye.
---
Français : Merci beaucoup.
Dioula   : N b'i fo kosɛbɛ.
---
Français : Je m'appelle Dorcas.
Dioula   : N tɔgɔ ko Dɔrɔkasi.
---


In [27]:
from datasets import Dataset, concatenate_datasets
import pandas as pd

# Créer un petit dataset de corrections
corrections = [
    {"translation": {"fr": "Bonjour.", "dyu": "A ni sogoma."}},
    {"translation": {"fr": "Bonjour !", "dyu": "A ni sogoma!"}},
    {"translation": {"fr": "Bonjour, comment vas-tu ?", "dyu": "A ni sogoma, i ka kɛnɛ wa?"}},
    {"translation": {"fr": "Bon après-midi.", "dyu": "A ni télé."}},
    {"translation": {"fr": "Bonsoir.", "dyu": "A ni wula."}},
    {"translation": {"fr": "Bonsoir, comment vas-tu ?", "dyu": "A ni wula, i ka kɛnɛ wa?"}},
    {"translation": {"fr": "Bonne nuit.", "dyu": "A ni su."}},
]

# Répéter 20 fois pour renforcer l'apprentissage
corrections = corrections * 20

dataset_corrections = Dataset.from_list(corrections)
print(f"✅ {len(dataset_corrections)} exemples de corrections créés")

# Ajouter au dataset d'entraînement existant
train_corrige = concatenate_datasets([
    dataset["train"],
    dataset_corrections
])
print(f"Dataset train original : {len(dataset['train'])} exemples")
print(f"Dataset train corrigé  : {len(train_corrige)} exemples")

✅ 140 exemples de corrections créés


ValueError: The features can't be aligned because the key translation of features {'translation': {'dyu': Value('string'), 'fr': Value('string')}} has unexpected type - {'dyu': Value('string'), 'fr': Value('string')} (expected either Translation(languages=['dyu', 'fr']) or Value("null").

In [28]:
from datasets import Dataset, concatenate_datasets
from datasets.features import Translation, Features

# Définir le bon format
features = Features({
    "ID": datasets.Value("string"),
    "translation": Translation(languages=["dyu", "fr"])
})

# Créer les corrections avec le bon format
corrections = [
    {"ID": f"correction_{i}", "translation": t}
    for i, t in enumerate([
        {"fr": "Bonjour.", "dyu": "A ni sogoma."},
        {"fr": "Bonjour !", "dyu": "A ni sogoma!"},
        {"fr": "Bonjour, comment vas-tu ?", "dyu": "A ni sogoma, i ka kɛnɛ wa?"},
        {"fr": "Bon après-midi.", "dyu": "A ni télé."},
        {"fr": "Bon après-midi, comment vas-tu ?", "dyu": "A ni télé, i ka kɛnɛ wa?"},
        {"fr": "Bonsoir.", "dyu": "A ni wula."},
        {"fr": "Bonsoir, comment vas-tu ?", "dyu": "A ni wula, i ka kɛnɛ wa?"},
        {"fr": "Bonne nuit.", "dyu": "A ni su."},
    ] * 20)
]

dataset_corrections = Dataset.from_list(corrections, features=features)

# Fusionner avec le dataset original
train_corrige = concatenate_datasets([
    dataset["train"],
    dataset_corrections
])
print(f"✅ Dataset corrigé : {len(train_corrige)} exemples")

NameError: name 'datasets' is not defined

In [29]:
import datasets
from datasets import Dataset, concatenate_datasets
from datasets.features import Translation, Features

# Définir le bon format
features = Features({
    "ID": datasets.Value("string"),
    "translation": Translation(languages=["dyu", "fr"])
})

# Créer les corrections avec le bon format
corrections = [
    {"ID": f"correction_{i}", "translation": t}
    for i, t in enumerate([
        {"fr": "Bonjour.", "dyu": "A ni sogoma."},
        {"fr": "Bonjour !", "dyu": "A ni sogoma!"},
        {"fr": "Bonjour, comment vas-tu ?", "dyu": "A ni sogoma, i ka kɛnɛ wa?"},
        {"fr": "Bon après-midi.", "dyu": "A ni télé."},
        {"fr": "Bon après-midi, comment vas-tu ?", "dyu": "A ni télé, i ka kɛnɛ wa?"},
        {"fr": "Bonsoir.", "dyu": "A ni wula."},
        {"fr": "Bonsoir, comment vas-tu ?", "dyu": "A ni wula, i ka kɛnɛ wa?"},
        {"fr": "Bonne nuit.", "dyu": "A ni su."},
    ] * 20)
]

dataset_corrections = Dataset.from_list(corrections, features=features)

# Fusionner avec le dataset original
train_corrige = concatenate_datasets([
    dataset["train"],
    dataset_corrections
])
print(f"✅ Dataset corrigé : {len(train_corrige)} exemples")

✅ Dataset corrigé : 8225 exemples


In [31]:
# Préparer le nouveau dataset
dataset_corrige_complet = datasets.DatasetDict({
    "train": train_corrige,
    "validation": dataset["validation"],
    "test": dataset["test"]
})

tokenizer3.src_lang = "fra_Latn"

def preprocess_nllb(exemples):
    inputs = [ex["fr"] for ex in exemples["translation"]]
    targets = [ex["dyu"] for ex in exemples["translation"]]
    model_inputs = tokenizer3(
        inputs, max_length=128, truncation=True, padding="max_length"
    )
    labels = tokenizer3(
        text_target=targets, max_length=128, truncation=True, padding="max_length"
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

dataset_nllb_corrige = dataset_corrige_complet.map(preprocess_nllb, batched=True)
print("✅ Données prêtes !")

# Entraînement court — juste 1 epoch pour intégrer les corrections
training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    num_train_epochs=1,  # ← 1 seule epoch suffit
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    fp16=True,
    optim="adafactor",
    predict_with_generate=False,
)

trainer = Seq2SeqTrainer(
    model=model3,
    args=training_args,
    train_dataset=dataset_nllb_corrige["train"],
    eval_dataset=dataset_nllb_corrige["validation"],
    data_collator=data_collator,
)

print(" Entraînement correctif...")
trainer.train()

model3.save_pretrained(output_dir)
tokenizer3.save_pretrained(output_dir)
print(" Corrections intégrées et modèle sauvegardé !")

Map:   0%|          | 0/8225 [00:00<?, ? examples/s]

✅ Données prêtes !
 Entraînement correctif...


Epoch,Training Loss,Validation Loss
1,29.640999,7.310893


 Corrections intégrées et modèle sauvegardé !


In [32]:
salutations = [
    "Bonjour.",
    "Bonjour, comment vas-tu ?",
    "Bon après-midi.",
    "Bonsoir.",
    "Bonne nuit.",
]

print("=== Test salutations ===\n")
for phrase in salutations:
    print(f"Français : {phrase}")
    print(f"Dioula   : {traduire_dioula(phrase)}")
    print("---")

=== Test salutations ===

Français : Bonjour.
Dioula   : Bonjour
---
Français : Bonjour, comment vas-tu ?
Dioula   : Bonjour, i be cogo di?
---
Français : Bon après-midi.
Dioula   : Dugutiɲɛ ɲuman.
---
Français : Bonsoir.
Dioula   : Sɔgɔmaɲuman.
---
Français : Bonne nuit.
Dioula   : Suuɲuman.
---


In [34]:
# Beaucoup plus de répétitions cette fois
corrections_renforcees = [
    {"ID": f"corr_{i}", "translation": t}
    for i, t in enumerate([
        {"fr": "Bonjour.", "dyu": "A ni sogoma."},
        {"fr": "Bonjour !", "dyu": "A ni sogoma!"},
        {"fr": "Bonjour, comment vas-tu ?", "dyu": "A ni sogoma, i ka kɛnɛ wa?"},
        {"fr": "Bonjour à tous.", "dyu": "A ni sogoma, an bɛɛ."},
        {"fr": "Bon après-midi.", "dyu": "A ni télé."},
        {"fr": "Bon après-midi, comment vas-tu ?", "dyu": "A ni télé, i ka kɛnɛ wa?"},
        {"fr": "Bonsoir.", "dyu": "A ni wula."},
        {"fr": "Bonsoir, comment vas-tu ?", "dyu": "A ni wula, i ka kɛnɛ wa?"},
        {"fr": "Bonne nuit.", "dyu": "A ni su."},
        {"fr": "Bonne nuit à tous.", "dyu": "A ni su, an bɛɛ."},
    ] * 100)  # ← 100 fois au lieu de 20
]

dataset_corrections2 = Dataset.from_list(corrections_renforcees, features=features)

train_corrige2 = concatenate_datasets([
    dataset["train"],
    dataset_corrections2
])
print(f" Dataset renforcé : {len(train_corrige2)} exemples")

 Dataset renforcé : 9065 exemples


In [35]:
dataset_corrige2_complet = datasets.DatasetDict({
    "train": train_corrige2,
    "validation": dataset["validation"],
    "test": dataset["test"]
})

dataset_nllb_corrige2 = dataset_corrige2_complet.map(preprocess_nllb, batched=True)
print(" Données prêtes !")

training_args2 = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    fp16=True,
    optim="adafactor",
    predict_with_generate=False,
)

trainer2 = Seq2SeqTrainer(
    model=model3,
    args=training_args2,
    train_dataset=dataset_nllb_corrige2["train"],
    eval_dataset=dataset_nllb_corrige2["validation"],
    data_collator=data_collator,
)

print(" Entraînement renforcé...")
trainer2.train()

model3.save_pretrained(output_dir)
tokenizer3.save_pretrained(output_dir)
print(" Terminé et sauvegardé !")

Map:   0%|          | 0/9065 [00:00<?, ? examples/s]

 Données prêtes !
 Entraînement renforcé...


Epoch,Training Loss,Validation Loss
1,29.514387,7.306071
2,29.487502,7.304333


 Terminé et sauvegardé !


In [36]:
print("=== Test salutations renforcé ===\n")
for phrase in salutations:
    print(f"Français : {phrase}")
    print(f"Dioula   : {traduire_dioula(phrase)}")
    print("---")

=== Test salutations renforcé ===

Français : Bonjour.
Dioula   : Aw ni sogoma.
---
Français : Bonjour, comment vas-tu ?
Dioula   : Aw ni sogoma, i ka kɛnɛ wa?
---
Français : Bon après-midi.
Dioula   : N ka wulafɛ.
---
Français : Bonsoir.
Dioula   : Aw ni su.
---
Français : Bonne nuit.
Dioula   : Suu ɲuman.
---


# Tester le modèle APRÈS fine-tuning
from transformers import MarianMTModel, MarianTokenizer

# Charger le modèle de traduction
model_name = "Helsinki-NLP/opus-mt-tc-big-fr-en"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

phrases_test = [
    "Bonjour, comment vas-tu ?",
    "Je veux apprendre le Dioula.",
    "L'eau est froide."
]

def traduire(texte):
    inputs = tokenizer(texte, return_tensors="pt", padding=True)
    outputs = model.generate(**inputs)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print("=== Traductions APRÈS fine-tuning ===\n")

for phrase in phrases_test:
    print(f"Français  : {phrase}")
    print(f"Dioula    : {traduire(phrase)}")
    print("---")

### How to find a new dataset on Hugging Face

1.  **Visit the Hugging Face Datasets Hub**: Go to [huggingface.co/datasets](https://huggingface.co/datasets).
2.  **Use the Search Bar**: Type in keywords like "French Dioula", "Bambara", "translation", or specific dataset names if you know them.
3.  **Filter by Languages**: Look for filters on the left sidebar to narrow down by source and target languages (e.g., "fr" for French, "dyu" for Dioula/Bambara).
4.  **Check Dataset Cards**: Click on potential datasets to view their "Dataset Card" (README). This card usually contains information about the dataset's content, size, intended use, and language pairs.
5.  **Look for `load_dataset` examples**: The dataset card often provides a direct `load_dataset` command you can copy and paste.

Once you've identified a dataset, you can load it using the `load_dataset` function, as shown below.